In [ ]:
import os
import pickle
import numpy as np

import scanpy as sc
import mudata

from scenicplus.data_wrangling.adata_cistopic_wrangling import process_multiome_data

In [ ]:
data_dir = "/scenic_plus_vol/scenic/full_dataset"
cisTopic_obj_fname = f"{data_dir}/cistopic_obj.pkl"
GEX_anndata_fname = f"{data_dir}/GEX.h5ad"
bc_transform_func = lambda x: f'{x}___cisTopic'
out_file = f"{data_dir}/100_000_cells_scplus_pipeline_out/ACC_GEX.h5mu"

os.mkdir(f"{data_dir}/100_000_cells_scplus_pipeline_out/")

In [ ]:
GEX_anndata = mudata.read(GEX_anndata_fname.__str__())

# downsample gene expression object to 100,000 cells
n = 100_000
sampled_cells = np.random.choice(GEX_anndata.obs.index.values, size=n, replace=False)
GEX_anndata = GEX_anndata[sampled_cells, :]
sc.write(f"{data_dir}/100_000_cells_scplus_pipeline_out/GEX.h5ad", GEX_anndata)
# del GEX_anndata

In [ ]:
# downaample atac object to 100,000 cells
with open(cisTopic_obj_fname, "rb") as f:
    cisTopic_obj = pickle.load(f)

sampled_cells = [f"{x}___cisTopic" for x in sampled_cells]
cisTopic_obj.subset(sampled_cells)
pickle.dump(cisTopic_obj,
            open(f"{data_dir}/100_000_cells_scplus_pipeline_out/cistopic_obj.pkl", "wb"))
# del cisTopic_obj

In [ ]:
# manually process rna+atac into combined object
# don't need to run this step as it's also part of the snakemake pipeline
mdata = process_multiome_data(
    GEX_anndata=GEX_anndata,
    cisTopic_obj=cisTopic_obj,
    use_raw_for_GEX_anndata=True,
    bc_transform_func=bc_transform_func)
mdata.write_h5mu(out_file)